<a href="https://colab.research.google.com/github/Squad-Nina-da-Hora/wmc-desafio-previsao-demencia/blob/main/analise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---
# **Análise de Risco de Alzheimer**
---


🎯 **Objetivo:**  Prever sinais de demência, através de informações clínicas e demográficas de pacientes com potencial risco de Alzheimer (OASIS).


---


Desafio Estatística com Python - Classificação

Squad Nina da Hora | Bootcamp Data Analytics 2026.1

## 1. Configurações Iniciais

Variáveis da base de dados:

- `Age`: Idade do paciente (numérico) 
- `Sex`: Gênero (F: feminino, M: masculino) 
- `EDUC`: Anos de escolaridade (numérico) 
- `SES`: Status socioeconômico (1 a 5) 
- `MMSE`: Escore do Mini Exame do Estado Mental (0 a 30) 
- `CDR`: Clinical Dementia Rating (0 a 3) 
- `eTIV`: Volume intracraniano estimado 
- `nWBV`: Proporção de volume cerebral normalizado 
- `ASF`: Fator de escala anatômica 
- `Group` (alvo): Classificação do paciente
  - `Nondemented` - será tratada para variável binária 0
  - `Demented` e `Converted` - serão tratadas para variável binária 1

In [ ]:
# ==============================
# IMPORTACOES
# ==============================

import kagglehub    # Para baixar datasets do Kaggle
import numpy as np  # Para operacoes numericas e arrays
import pandas as pd  # Para manipulaco e analise de data frames
from IPython.display import display, Markdown

# Bibliotecas para criacao de graficos
import seaborn as sns
import matplotlib.pyplot as plt
import missingno as msno

# Bibliotecas para criacao de modelos de ML
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedGroupKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils import resample

# Carregamento da base de dados
arquivo = 'oasis_longitudinal'
url = f'{kagglehub.dataset_download("jboysen/mri-and-alzheimers")}/{arquivo}.csv'
df = pd.read_csv(url).drop(columns=['Subject ID', 'MRI ID', 'Visit', 'MR Delay', 'Hand'])

In [ ]:
# ==============================
# VARIAVEIS PARA REUTILIZACAO
# ==============================

alvo = 'Group'

# Configuracos visuais dos graficos
paleta = 'flare'
cores = sns.color_palette(paleta, n_colors=2)

In [ ]:
# ==============================
# TRATAMENTO INICIAL
# ==============================

# 1. Renomeia algumas colunas
df.rename(columns={
  'Group': 'Demented',
  'M/F': 'Sex'
}, inplace=True)
# 2. Converte a coluna 'Demented' para bool
df['Demented'] = df['Demented'] != 'Nondemented'
# 3. Converte as colunas especificas para categorias
vars_categoricas = ['Sex', 'SES']
for cat in vars_categoricas:
  df[cat] = df[cat].astype('category')

df.head()

## 2. Análise Exploratória


In [ ]:
df.describe()

In [ ]:
df.describe(include=['object', 'category', 'bool'])

In [ ]:
contagem_nulos = df.isnull().sum()

datadict = pd.DataFrame(df.dtypes)
datadict.columns = ['tipos']
datadict['nulos'] = contagem_nulos
datadict['%_nulos'] = ((contagem_nulos / df.shape[0]) * 100).round(2)
datadict['unicos'] = df.nunique()
datadict

## 3. Modelo de ML